In [77]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

start_date = datetime(2026, 1, 1)
clean_logs = []

np.random.seed(42)

for day in range(150):
    current_date = start_date + timedelta(days=day)
    day_of_week = current_date.weekday() # 0 = Monday, 6 = Sunday
    
    # Mid-week base severity is around 4 out of 10
    base_severity = 4
    
    # Force a major spike on Sundays and Mondays
    if day_of_week in [0, 6]:
        base_severity += 4  # Boost to 8
    else:
        # Add normal variation (-1, 0, or 1)
        base_severity += np.random.choice([-1, 0, 1])
        
    # Lock the boundaries to your 1-10 scale
    final_severity = max(1, min(10, base_severity))
    
    clean_logs.append({
        "timestamp": current_date,
        "severity": int(final_severity)
    })

study_df = pd.DataFrame(clean_logs)
sdf = pd.DataFrame(clean_logs)

In [78]:
# ensure the timeline flows naturally day-by-day
study_df = study_df.sort_values('timestamp').reset_index(drop=True) #sort by chronilogical order
study_df['day_of_week'] = study_df['timestamp'].dt.dayofweek #add the day of the week col

# compute features globally across the true calendar timeline
study_df['severity_yesterday'] = study_df['severity'].shift(1)
study_df['severity_2d_ago'] = study_df['severity'].shift(2)

# calculate rolling volatility over the literal consecutive days
study_df['rolling_volatility_3d'] = study_df['severity'].rolling(window=3, min_periods=1).std()

# now group by day of the week to analyze how your features behave on those days
volatility_profile = study_df.groupby('day_of_week')['rolling_volatility_3d'].mean()
print(volatility_profile)

day_of_week
0    2.254415
1    2.281908
2    2.380403
3    0.692715
4    0.658960
5    0.759168
6    2.361168
Name: rolling_volatility_3d, dtype: float64


In [79]:
#drop the rows with nan values at the top of the dataframe
ml_df = study_df.dropna().copy()

#define the input features X and what we want to predict Y
feature_cols = ['severity_yesterday', 'severity_2d_ago', 'rolling_volatility_3d']
X = ml_df[feature_cols]
y = ml_df['severity']

#chronological split point (train on 80% and test on 20%)
split_index = int(len(ml_df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training shapes: {X_train.shape}, Testing shapes: {X_test.shape}")


Training shapes: (118, 3), Testing shapes: (30, 3)


In [80]:
from sklearn.tree import DecisionTreeRegressor

#init the model
#max_depth=3 keeps the flowchart simple, it learns the big pattern without over complication 
model = DecisionTreeRegressor(max_depth = 3, random_state=42)

#train the model using the training sets
model.fit(X_train, y_train)

print("Model training complete, the flowchart has been built")

Model training complete, the flowchart has been built


In [81]:
#use the prediction now that the model is trained
predictions = model.predict(X_test)

from sklearn.metrics import mean_absolute_error

#test the predictions against the actual result, then print
mae = mean_absolute_error(y_test, predictions)
print(mae)

0.4560606060606061


In [82]:
#compare the exact data against the predictions
comp_df = pd.DataFrame()
comp_df['actual'] = y_test.values
comp_df['predicted'] = predictions
comp_df['error'] = comp_df['actual'] - comp_df['predicted']
print(comp_df.head(15))

    actual  predicted     error
0        5   4.333333  0.666667
1        3   3.636364 -0.636364
2        8   8.000000  0.000000
3        8   8.000000  0.000000
4        4   3.636364  0.363636
5        3   3.636364 -0.636364
6        3   4.153846 -1.153846
7        3   3.000000  0.000000
8        3   3.000000  0.000000
9        8   8.000000  0.000000
10       8   8.000000  0.000000
11       5   4.833333  0.166667
12       3   3.636364 -0.636364
13       3   3.636364 -0.636364
14       3   3.000000  0.000000


In [85]:
# ensure the timeline flows naturally day-by-day
sdf = sdf.sort_values('timestamp').reset_index(drop=True) #sort by chronilogical order
sdf['day_of_week'] = sdf['timestamp'].dt.dayofweek #add the day of the week col

# compute features globally across the true calendar timeline
sdf['severity_lag_1'] = sdf['severity'].shift(1)
sdf['severity_lag_2'] = sdf['severity'].shift(2)
sdf['severity_lag_3'] = sdf['severity'].shift(3)
sdf['severity_lag_4'] = sdf['severity'].shift(4)
sdf['severity_lag_5'] = sdf['severity'].shift(5)
sdf['severity_lag_6'] = sdf['severity'].shift(6)
sdf['severity_lag_7'] = sdf['severity'].shift(7)


# calculate rolling volatility over the literal consecutive days
sdf['rolling_volatility_7d'] = sdf['severity'].rolling(window=7, min_periods=1).std()

# now group by day of the week to analyze how your features behave on those days
volatility_profile = sdf.groupby('day_of_week')['rolling_volatility_7d'].mean()
print(volatility_profile)
print(sdf.head(10))

day_of_week
0    2.062617
1    2.065243
2    2.052949
3    2.050240
4    2.020165
5    2.016381
6    2.057551
Name: rolling_volatility_7d, dtype: float64
   timestamp  severity  day_of_week  severity_lag_1  severity_lag_2  \
0 2026-01-01         5            3             NaN             NaN   
1 2026-01-02         3            4             5.0             NaN   
2 2026-01-03         5            5             3.0             5.0   
3 2026-01-04         8            6             5.0             3.0   
4 2026-01-05         8            0             8.0             5.0   
5 2026-01-06         5            1             8.0             8.0   
6 2026-01-07         3            2             5.0             8.0   
7 2026-01-08         3            3             3.0             5.0   
8 2026-01-09         5            4             3.0             3.0   
9 2026-01-10         4            5             5.0             3.0   

   severity_lag_3  severity_lag_4  severity_lag_5  severity_lag_

In [91]:
#drop the rows with nan values at the top of the dataframe
ml_sdf = sdf.dropna().copy()

#define the input features X and what we want to predict Y
feature_cols = ['severity_lag_1', 'severity_lag_2', 'severity_lag_3', 'severity_lag_4','severity_lag_5', 'severity_lag_6',
                'severity_lag_7', 'rolling_volatility_7d',]
X_7d = ml_sdf[feature_cols]
Y_7d = ml_sdf['severity']

#chronological split point (train on 80% and test on 20%)
split_index = int(len(ml_sdf) * 0.8)

X_7d_train, X_7d_test = X_7d.iloc[:split_index], X_7d.iloc[split_index:]
Y_7d_train, Y_7d_test = Y_7d.iloc[:split_index], Y_7d.iloc[split_index:]

print(f"Training shapes: {X_7d_train.shape}, Testing shapes: {X_7d_test.shape}")

Training shapes: (114, 8), Testing shapes: (29, 8)


In [95]:
#make a new model 
model_7d = DecisionTreeRegressor(max_depth=3, random_state=42)

#train the model on the 80 20
model_7d.fit(X_7d_train, Y_7d_train)

print("Model has been trained")

#predict the outcome with model on 20%
predictions_7d = model_7d.predict(X_7d_test)

#compare the results with the mae
mae_7d = mean_absolute_error(predictions_7d, Y_7d_test)
print(mae_7d)


Model has been trained
0.610594974025101


In [99]:
optimized_features = ['severity_lag_1', 'severity_lag_2', 'severity_lag_7']

X_train_opt = X_7d_train[optimized_features]
X_test_opt = X_7d_test[optimized_features]

model_opt = DecisionTreeRegressor(max_depth=3, random_state=42)
model_opt.fit(X_train_opt, Y_7d_train)

predictions_opt = model_opt.predict(X_test_opt)

mae_opt = mean_absolute_error(predictions_opt, Y_7d_test)
print(mae_opt)

0.7281167108753317


In [100]:
comp_opt_df = pd.DataFrame()
comp_opt_df['actual'] = y_7d_test.values
comp_opt_df['predicted'] = predictions_opt
comp_opt_df['error'] = comp_opt_df['actual'] - comp_opt_df['predicted']
print(comp_opt_df.head(15))

    actual  predicted     error
0        3   3.961538 -0.961538
1        8   8.000000  0.000000
2        8   8.000000  0.000000
3        4   4.500000 -0.500000
4        3   4.076923 -1.076923
5        3   3.961538 -0.961538
6        3   4.222222 -1.222222
7        3   4.076923 -1.076923
8        8   8.000000  0.000000
9        8   8.000000  0.000000
10       5   3.961538  1.038462
11       3   4.500000 -1.500000
12       3   4.076923 -1.076923
13       3   4.076923 -1.076923
14       5   4.076923  0.923077


In [101]:
def prepare_real_user_timeline(df):
    # 1. Convert timestamp column to actual datetime objects if they aren't already
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # 2. Set the timestamp as the index so pandas can see the calendar grid
    df = df.set_index('timestamp')
    
    # 3. Reindex the dataframe to a strict daily calendar grid from the minimum date to the maximum date
    # This automatically inserts blank rows with NaN for any dates the user skipped!
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df_reindexed = df.reindex(full_range)
    
    # 4. Use Forward Fill to safely handle missing entries so our lags align perfectly in time
    df_clean = df_reindexed.ffill()
    
    # 5. Bring the timestamp back out as a normal column and reset the index
    df_clean = df_clean.reset_index().rename(columns={'index': 'timestamp'})
    
    return df_clean

# Create a tiny broken sample dataset with a missing day (Jan 2nd is gone)
broken_data = pd.DataFrame([
    {"timestamp": "2026-01-01", "severity": 4},
    {"timestamp": "2026-01-03", "severity": 6}
])

# Run it through our real-world pipeline function
reindexed_data = prepare_real_user_timeline(broken_data)
print(reindexed_data)

   timestamp  severity
0 2026-01-01       4.0
1 2026-01-02       4.0
2 2026-01-03       6.0
